In [2]:
library(scop)
library(Seurat)
library(ggplot2)
library(dplyr)
setwd("/")
set.seed(4180)
###### colors#######
cols <- c('#92a5d1','#C9DCC4','#D9B9D4','#E83945',
 '#7C9895','#c5dff4','#DAA87C','#FBDFE2',
 '#F4EEAC','#88CEE6','#E3E457')
pal <- colorRampPalette(cols)

Registered S3 method overwritten by 'scop':
  method             from  
  FoldChange.default Seurat

          ⬢          .        ⬡             ⬢     .
                     _____ _________  ____
                    / ___// ___/ __ ./ __ .
                   (__  )/ /__/ /_/ / /_/ /
                  /____/ .___/.____/ .___/
                                  /_/
      ⬢               .      ⬡        .          ⬢

------------------------------------------------------------
Version: 0.8.7 (2026-03-31 update)
Website: https://mengxu98.github.io/scop/

Python environment initialization is disabled
To enable it, set: options(scop_env_init = TRUE)

The message can be suppressed by: 
  suppressPackageStartupMessages(library(scop))
  or options(log_message.verbose = FALSE)
------------------------------------------------------------

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:


In [1]:
subtype.analysis <- function(seurat.obj, type.choose = NULL) {
 data.list <- list()
 type.choose <- type.choose
 for (i in 1:length(type.choose)) {
 print(type.choose[i])
 data <- subset(seurat.obj, celltype %in% type.choose[i])
 data <- subset(data, orig.ident %in% names(which(table(data$orig.ident) >= 10)))
 data[["RNA"]] <- CreateAssay5Object(GetAssayData(data,
 assay = "RNA",
 layer = "counts"
 ))
 data[["RNA"]] <- split(data[["RNA"]], f = data$orig.ident)
 data <- data %>%
 NormalizeData(verbose = FALSE) %>%
 FindVariableFeatures(verbose = FALSE) %>%
 ScaleData(vars.to.regress = c("percent.mt", "percent.rb", "percent.hsp")) %>%
 RunPCA(verbose = FALSE) %>%
 IntegrateLayers(.,
 method = HarmonyIntegration,
 orig.reduction = "pca",
 new.reduction = "harmony",
 verbose = FALSE
 ) %>%
 FindNeighbors(., reduction = "harmony", dims = 1:20, verbose = FALSE) %>%
 FindClusters(., resolution = 0.2, verbose = FALSE) %>%
 RunUMAP(.,
 dims = 1:20,
 reduction = "harmony",
 reduction.name = "umap",
 verbose = FALSE
 )
 meta <- data@meta.data
 total_samples <- length(unique(meta$id))

 # Count cells per Cluster per Sample
 cluster_sample_stats <- meta %>%
 group_by(seurat_clusters, id) %>%
 summarise(n_cells = n(), .groups = "drop") %>%
 # Flag whether the Cluster is "abundant" in that sample (>10 cells)
 mutate(is_abundant = ifelse(n_cells > 10, 1, 0)) %>%
 # Summarize by Cluster
 group_by(seurat_clusters) %>%
 summarise(
 total_cells = sum(n_cells),
 avg_cells_per_sample = total_cells / total_samples,
 n_samples_abundant = sum(is_abundant), # number of samples with cell count > 10
 prevalence_pct = (n_samples_abundant / total_samples) * 100, # proportion %
 .groups = "drop"
 )

 # 2. Apply the filtering logic
 # Keep if: mean >= 5 OR (cell count > 10 in >= 20% of samples)
 keep_clusters_df <- cluster_sample_stats %>%
 filter(avg_cells_per_sample >= 5 | prevalence_pct >= 20)

 keep_clusters <- keep_clusters_df$seurat_clusters

 # Inspect removed Clusters and the reason
 removed_clusters_df <- anti_join(cluster_sample_stats, keep_clusters_df, by = "seurat_clusters")
 if (nrow(removed_clusters_df) > 0) {
 print("Removed Clusters (Low Avg AND Low Prevalence):")
 print(removed_clusters_df)
 } else {
 print("No clusters removed based on these criteria.")
 }
 data <- subset(data, subset = seurat_clusters %in% keep_clusters)
 data$subtype <- paste(type.choose[i], "-",
 as.numeric(data$seurat_clusters),
 sep = ""
 )
 data$subtype <- factor(data$subtype, levels = paste(type.choose[i],
 sort(unique(as.numeric(data$seurat_clusters))),
 sep = "-"
 ))
 data <- JoinLayers(data)
 data.list[[i]] <- data
 }
 names(data.list) <- type.choose
 return(data.list)
}

In [5]:
mHeart <- readRDS("data/mHeart.Rds")

In [6]:
celltypenames <- levels(mHeart$celltype)
celltypenames

[1] "vCM"        "aCM"        "FB"         "EC"         "EndoCC"    
 [6] "LEC"        "SMC"        "Pericyte"   "Adipocyte"  "Neuronal"  
[11] "T"          "B"          "Macrophage" "DC"

In [7]:
subsc.list <- subtype.analysis(mHeart,
 type.choose=celltypenames[-1])

[1] "aCM"


Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -1.7276”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.32109”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  1.0533e-30”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“There are other near singularities as well. 0.031008”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -2.0294”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.30103”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  2.0078e-16”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse us

[1] "Removed Clusters (Low Avg AND Low Prevalence):"
# A tibble: 1 × 5
  seurat_clusters total_cells avg_cells_per_sample n_samples_abundant
  <fct>                 <int>                <dbl>              <dbl>
1 4                        26                 1.73                  0
# ℹ 1 more variable: prevalence_pct <dbl>
[1] "FB"


Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -2.7156”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.49902”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  1.0124e-15”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“There are other near singularities as well. 0.090619”
Regressing out percent.mt, percent.rb, percent.hsp

Centering and scaling data matrix



[1] "No clusters removed based on these criteria."
[1] "EC"


...

[1] "Removed Clusters (Low Avg AND Low Prevalence):"
# A tibble: 2 × 5
  seurat_clusters total_cells avg_cells_per_sample n_samples_abundant
  <fct>                 <int>                <dbl>              <dbl>
1 1                        54                 4.91                  1
2 2                        29                 2.64                  1
# ℹ 1 more variable: prevalence_pct <dbl>
[1] "Macrophage"


Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -2.4872”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.49742”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  1.1579e-15”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“There are other near singularities as well. 0.090619”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -2.0272”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.3196”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  0”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“There are other near singu

[1] "Removed Clusters (Low Avg AND Low Prevalence):"
# A tibble: 2 × 5
  seurat_clusters total_cells avg_cells_per_sample n_samples_abundant
  <fct>                 <int>                <dbl>              <dbl>
1 6                        53                 3.53                  0
2 7                        46                 3.07                  0
# ℹ 1 more variable: prevalence_pct <dbl>
[1] "DC"


Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“at  -1.2707”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“radius  0.00023781”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“all data on boundary of neighborhood. make span bigger”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“pseudoinverse used at -1.2707”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“neighborhood radius 0.015421”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“reciprocal condition number  1”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“There are other near singularities as well. 0.090619”
Warning message in simpleLoess(y, x, w, span, degree = degree, parametric = parametric, :
“zero-width neighborhood. make span b

[1] "Removed Clusters (Low Avg AND Low Prevalence):"
# A tibble: 1 × 5
  seurat_clusters total_cells avg_cells_per_sample n_samples_abundant
  <fct>                 <int>                <dbl>              <dbl>
1 1                        21                    3                  0
# ℹ 1 more variable: prevalence_pct <dbl>


In [8]:
saveRDS(subsc.list,"data/subsc.list.Rds")